## Creating a spatially interpolated surface using MRoS observation point data, station (LCD, HADS, WCC) point data, gridded IMERG pLP, and DEM elevation.

In [ ]:
"""
IDW interpolation for precipitation phase (rain/snow/mix).

To accurately predict phase, especially at air temperatures near freezing, we need to consider assimilating various data types (MRoS point observations, station point observations, IMERG gridded/point-level probabilities of liquid precip). 
We need to spatially interpolate MRoS observation point data across a domain grid to create an hourly ground truth phase surface...


1) Loads and harmonizes 
    a) MRoS observations: lat/long, timestamp in UTC, phase
        Provides ground-truth phase data (rain, snow, mixed) with location (lat/long) and UTC timestamp. These are the primary interpolation targets for the phase surface.
    b) Gage Stations' predictors: lat/long, hourly timestamp, elevation, rh, temp_air, temp_dew, temp_wet
        Supplies continuous predictor variables (elevation, relative humidity, air temperature, dew point temperature, wet bulb temperature) at known locations and hourly timestamps. Used as covariates in interpolation.
    c) IMERG GPM pLP: lat/long as x/y, half-hourly timestamp (map from local time zone of station to UTC), probability of LP
        Half-hourly satellite-derived probability of liquid precipitation gridded dataset. Sampled at station/observation locations (lat/long in projected coordinates) and temporally matched to UTC. Used as an additional spatial predictor to improve phase classification accuracy.
    d) elevation DEM: raster of elevation by pixel
        Pulled in at MRoS observation and IMERG pLP coordinates to provide a consistent elevation predictor across all datasets.

2) Runs IDW using cKDTree on:
    a) Phase points from MRoS data
        Interpolates discrete phase observations into a continuous phase probability surface.
    b) predictor variables from station data
        Interpolates continuous predictors to create gridded covariate layers for mapping and potential model fitting.

3) Visualizes on a map

"""

import os, glob, warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from scipy.spatial import cKDTree
from dateutil import parser
from zoneinfo import ZoneInfo

try:
    import rasterio
    from rasterio.warp import transform as rio_transform
except Exception:
    rasterio = None


# ----------------------
# PATHS 
# ----------------------
BASE = Path.cwd()

PATHS = {
    "mros_parquet": BASE / "Data/observations/wy25_mros_obs.parquet",
    "hads_csv":     BASE / "Data/Stations/hads_20241001_20250531.csv",
    "lcd_csv":      BASE / "Data/Stations/lcd_20241001_20250531.csv",
    "wcc_csv":      BASE / "Data/Stations/wcc_20241001_20250531.csv",
    "station_meta": BASE / "Data/Stations/station_metadata_20241001_20250531.csv",
    "gpm_glob":     str(BASE / "Data/IMERG/imerg_data-20250731T220028Z-1-001/imerg_data/gpm_imerg_*.parquet"),
    "elev_tif":     str("C:/Users/EmmaGolub/Desktop/MRoS_local/local_data/DEM_AOI_TNM_10m.tif") # sourcing from personal drive because too large to push to repo
}


In [ ]:
# ----------------------------
# OPEN AND CHECK DATA FORMATS 
# ----------------------------

# MRoS parquet
mros_df = pd.read_parquet(os.path.join(BASE / "Data/observations/wy25_mros_obs.parquet"))
print("MRoS Parquet:")
print(mros_df.head(), "\n")

# GPM IMERG parquet
gpm_df = pd.read_parquet(os.path.join(BASE / "Data/IMERG/imerg_data-20250731T220028Z-1-001/imerg_data/gpm_imerg_2024-10-11.parquet"))
print("GPM IMERG Parquet:")
print(gpm_df.head(), "\n")

# Station CSV
hads_df = pd.read_csv(os.path.join(BASE / "Data/Stations/hads_20241001_20250531.csv"))
print("HADS CSV:")
print(hads_df.head(), "\n")

lcd_df = pd.read_csv(os.path.join(BASE / "Data/Stations/lcd_20241001_20250531.csv"))
print("LCD CSV:")
print(lcd_df.head(), "\n")

wcc_df = pd.read_csv(os.path.join(BASE / "Data/Stations/wcc_20241001_20250531.csv"))
print("WCC CSV:")
print(wcc_df.head(), "\n")

meta_df = pd.read_csv(os.path.join(BASE / "Data/Stations/station_metadata_20241001_20250531.csv"))
print("Metadata CSV:")
print(meta_df.head(), "\n")

In [ ]:
# ----------------------
# Initializations
# ----------------------

# Process a UTC calendar day only for testing (e.g., "2024-10-11"); set to None to auto-pick from data
TEST_DATE = "2024-10-11" # None

# IDW blending weights: 
RES_DEG = 0.01        # grid step (~1 km near 39N)
K_NEIGH = 12
POWER   = 2.0
ALPHA_MROS   = None #1.0    # source weights in IDW
ALPHA_STN    = None #0.6
ALPHA_IMERG = None #0.25

# ----------------------
# Functions
# ----------------------
def dem_bounds_lonlat(tif_path):
    with rasterio.open(tif_path) as src:
        b = src.bounds
        xs = np.array([b.left, b.right, b.right, b.left])
        ys = np.array([b.top,  b.top,   b.bottom, b.bottom])
        if str(src.crs).lower() != "epsg:4326":
            xs_ll, ys_ll = rio_transform(src.crs, "EPSG:4326", xs, ys)
        else:
            xs_ll, ys_ll = xs, ys
    return float(np.min(xs_ll)), float(np.min(ys_ll)), float(np.max(xs_ll)), float(np.max(ys_ll))


def bbox_filter(df, lon="lon", lat="lat", bbox=None):
    xmin, ymin, xmax, ymax = bbox
    return df[df[lon].between(xmin,xmax) & df[lat].between(ymin,ymax)].copy()


def parse_station_ts_to_utc_hour(ts_str, tz_str):
    """
    Parse a station timestamp to UTC hour.
    - If ts_str has tz info (e.g., '...Z' or '+/-HH:MM'), convert to UTC.
    - If unclear and tz_str present (e.g., 'Etc/GMT+8'), localize to that tz then convert to UTC.
    """
    ts = pd.to_datetime(ts_str, errors="coerce")
    if ts is pd.NaT:
        return pd.NaT

    # If timestamp carries tz info, just convert to UTC -> hourly
    if getattr(ts, "tzinfo", None) is not None:
        return ts.tz_convert("UTC").tz_localize(None).floor("H")

    # No tz info on the timestamp → require metadata timezone
    if pd.isna(tz_str) or not str(tz_str).strip():
        raise SystemExit(
            "ERROR: Station timestamp is naive (no timezone), and metadata timezone is missing.\n"
            f"Sample timestamp: {ts_str!r}\n"
            "Please fix station metadata 'timezone_lst' so we can localize to UTC."
        )

    # Try to localize using provided metadata timezone (e.g., 'Etc/GMT+8', 'America/Los_Angeles')
    try:
        tz = ZoneInfo(str(tz_str).strip())
    except Exception as e:
        raise SystemExit(
            "ERROR: Could not interpret station metadata timezone.\n"
            f"timezone_lst={tz_str!r}  Exception: {e}\n"
            f"Sample timestamp: {ts_str!r}\n"
            "Update 'timezone_lst' to a valid IANA zone, e.g. 'America/Los_Angeles'.\n"
            "Note: 'Etc/GMT+8' means UTC-8 (counterintuitive sign)."
        )
    
    ts_local = ts.tz_localize(tz)
    return ts_local.tz_convert("UTC").tz_localize(None).floor("H")


def idw_predict_grid_weighted(xy_deg, z, xs_deg, ys_deg, k=12, power=2.0, alpha=None):
    """
    Perform Inverse Distance Weighting (IDW) interpolation on lon/lat data.
    Parameters:
        xy_deg : Array of shape (n_points, 2) with source point coordinates [lon, lat] in degrees.
        z : Array of source point values (same length as xy_deg).
        xs_deg, ys_deg : Arrays of target grid coordinates (longitude and latitude) in degrees.
        k : Number of nearest neighbors to use for interpolation.
        power : Power parameter for IDW (higher → faster decay with distance).
        alpha : Array of additional weights for each source point (e.g., MRoS phase observations are more reliable, so would weight these hgiher).
    """

    if len(xy_deg) == 0:
        return np.full((len(ys_deg), len(xs_deg)), np.nan)
    # Compute mean latitude of source points
    lat0 = np.mean(xy_deg[:, 1])
    # Cosine scaling factor for longitude distances at mean latitude (lon degrees shrink in physical distance toward the poles)
    cx = np.cos(np.deg2rad(lat0))
    # Apply longitude scaling to the source point coordinates
    XY = np.column_stack([xy_deg[:, 0] * cx, xy_deg[:, 1]])
    # Build a target grid in the same scaled coordinate system
    gx, gy = np.meshgrid(xs_deg * cx, ys_deg)
    P = np.column_stack([gx.ravel(), gy.ravel()])  # Flattened grid points
    # Build a KD-tree for fast nearest-neighbor lookup among source points
    tree = cKDTree(XY)
    # Ensure k is not greater than number of available source points
    k = min(k, len(XY))
    # Query the k nearest neighbors for each target point
    # Returns distances (dists) and indices (idxs) into the source array
    dists, idxs = tree.query(P, k=k)
    # If k=1, force arrays into 2D shape 
    if k == 1:
        dists = dists[:, None]
        idxs = idxs[:, None]
    # Compute base IDW weights: 1 / distance^power
    base = 1.0 / (np.power(dists, power) + 1e-12)
    # If alpha weights are given, multiply them in (per neighbor)
    W = base if alpha is None else base * alpha[idxs]
    # Weighted sum of neighbor values (numerator)
    num = np.sum(W * z[idxs], axis=1)
    # Sum of weights (denominator)
    den = np.sum(W, axis=1)
    # Final IDW estimate = weighted sum / sum of weights
    return (num / den).reshape(len(ys_deg), len(xs_deg))


def sample_elev_points(tif_path, lons, lats):
    with rasterio.open(tif_path) as src:
        if str(src.crs).lower() != "epsg:4326":
            xs_r, ys_r = rio_transform("EPSG:4326", src.crs, lons, lats)
            coords = list(zip(xs_r, ys_r))
        else:
            coords = list(zip(lons, lats))
        vals = np.fromiter((v[0] for v in src.sample(coords)), dtype=float, count=len(coords))
    return vals

def sample_elev_grid(tif_path, xs, ys):
    with rasterio.open(tif_path) as src:
        if str(src.crs).lower() != "epsg:4326":
            LON, LAT = np.meshgrid(xs, ys)
            xs_r, ys_r = rio_transform("EPSG:4326", src.crs, LON.ravel(), LAT.ravel())
            coords = list(zip(xs_r, ys_r))
        else:
            coords = [(xx, yy) for yy in ys for xx in xs]
        vals = np.array(list(src.sample(coords))).reshape(len(ys), len(xs), -1)[...,0]
    return vals

def melt_imerg_wide(df):
    g = df.rename(columns={"x":"lon","y":"lat"}).copy()
    time_cols = []
    for c in g.columns:
        if c in ("lon","lat"): continue
        try: pd.to_datetime(c); time_cols.append(c)
        except: pass
    if not time_cols: return pd.DataFrame(columns=["time_hr","lon","lat","p_liquid"])
    long = g.melt(id_vars=["lon","lat"], value_vars=time_cols, var_name="time_str", value_name="p_liquid")
    long["time"] = pd.to_datetime(long["time_str"], utc=True, errors="coerce")
    long["time_hr"] = long["time"].dt.tz_localize(None).dt.floor("H")
    long["p_liquid"] = pd.to_numeric(long["p_liquid"], errors="coerce")/100.0
    long["p_liquid"] = long["p_liquid"].clip(0,1)
    long = long.dropna(subset=["time_hr","lon","lat","p_liquid"])
    # average two half-hours per hour per cell if both present
    return long.groupby(["time_hr","lon","lat"], as_index=False)["p_liquid"].mean()

def load_station_csv(path):
    df = pd.read_csv(path)
    df.columns = [c.lower() for c in df.columns]
    if "id" in df.columns: df.rename(columns={"id":"station_id"}, inplace=True)
    return df

In [ ]:
# ----------------------- AOI from DEM -----------------------
xmin, ymin, xmax, ymax = dem_bounds_lonlat(PATHS["elev_tif"])
AOI_BBOX = (xmin, ymin, xmax, ymax)
xs = np.arange(xmin, xmax + RES_DEG, RES_DEG)
ys = np.arange(ymin, ymax + RES_DEG, RES_DEG)
dem_grid = sample_elev_grid(PATHS["elev_tif"], xs, ys)


In [ ]:
# ----------------------- Load + clean MRoS -------------------
raw = pd.read_parquet(PATHS["mros_parquet"])
raw.columns = [c.lower() for c in raw.columns]
need = {"latitude","longitude","phase","time_submitted_utc","date_submitted_utc"}
if not need.issubset(raw.columns):
    raise SystemExit(f"[STOP] MRoS missing required fields: {need - set(raw.columns)}")

mros = raw.rename(columns={"latitude":"lat","longitude":"lon","phase":"phase_raw"}).copy()
dt_str = mros["date_submitted_utc"].astype(str).str.strip() + " " + mros["time_submitted_utc"].astype(str).str.strip()
mros["time"] = pd.to_datetime(dt_str, utc=True, errors="coerce")
mros["time_hr"] = mros["time"].dt.tz_localize(None).dt.floor("H")
mros["phase"] = mros["phase_raw"].astype(str).str.lower().str.strip()
mros["lon"] = pd.to_numeric(mros["lon"], errors="coerce")
mros["lat"] = pd.to_numeric(mros["lat"], errors="coerce")
mros = mros.dropna(subset=["lon","lat","time_hr"])
mros = bbox_filter(mros, "lon","lat", AOI_BBOX)[["lon","lat","phase","time_hr"]]
print(f"[INFO] MRoS rows in AOI: {len(mros)}")

In [ ]:
# ----------------------- Load stations --------
meta = pd.read_csv(PATHS["station_meta"])
meta.columns = [c.lower() for c in meta.columns]

st_frames = [load_station_csv(PATHS["hads_csv"]),
             load_station_csv(PATHS["lcd_csv"]),
             load_station_csv(PATHS["wcc_csv"])]
frames = [f for f in st_frames if f is not None]  # avoid ambiguous truth values
st_all = pd.concat(frames, ignore_index=True) if len(frames) > 0 else None

if (st_all is not None) and (meta is not None):
    # attach coords/elev/tz
    st_all = st_all.merge(
        meta.rename(columns={"id":"station_id"})[["station_id","lat","lon","elev","timezone_lst"]],
        on="station_id", how="left"
    )
    st_all = st_all.dropna(subset=["lat","lon"])
    st_all = bbox_filter(st_all, "lon","lat", AOI_BBOX)

    # per-row timestamp -> UTC hour using each station tz
    tcol = next((c for c in ["time","datetime","date_time","obstime"] if c in st_all.columns), None)
    if tcol:
        st_all["time_hr"] = st_all.apply(lambda r: parse_station_ts_to_utc_hour(r[tcol], r.get("timezone_lst", None)), axis=1)
    else:
        st_all["time_hr"] = pd.NaT

    # keep predictors for second IDW on station data
    for c in ["temp_air","temp_dew","temp_wet","rh","elev"]:
        if c in st_all.columns:
            st_all[c] = pd.to_numeric(st_all[c], errors="coerce")
    print(f"[INFO] Station rows in AOI: {len(st_all)}")
else:
    print("[INFO] No station data available or missing metadata.")
    st_all = None


In [ ]:
# ----------------------- Load IMERG  ------------
imerg = None
if TEST_DATE:
    day_file = str(BASE / "Data/IMERG/imerg_data-20250731T220028Z-1-001/imerg_data/gpm_imerg_{TEST_DATE}.parquet")
    if Path(day_file).exists():
        imerg = melt_imerg_wide(pd.read_parquet(day_file))
        imerg = bbox_filter(imerg, "lon","lat", AOI_BBOX)
        print(f"[INFO] IMERG rows in AOI for {TEST_DATE}: {len(imerg)}")
    else:
        print(f"[WARN] No IMERG file found for {TEST_DATE}: {day_file}")
else:
    # fallback: glob all
    gpm_files = glob.glob(PATHS["gpm_glob"])
    imerg = None
    if gpm_files:
        long_list = []
        for f in gpm_files:
            try:
                long_list.append(melt_imerg_wide(pd.read_parquet(f)))
            except Exception as e:
                print(f"[WARN] IMERG read failed: {f} ({e})")
        if len(long_list) > 0:
            imerg = pd.concat(long_list, ignore_index=True)
            imerg = bbox_filter(imerg, "lon","lat", AOI_BBOX)
            print(f"[INFO] IMERG rows in AOI: {len(imerg)}")


In [ ]:
# ----------------------- Choose an hour within TEST_DATE -----------------------
def hours(df):
    # Return the set of unique hourly timestamps in this dataframe.
    return set(pd.to_datetime(df["time_hr"]).dropna().unique()) if (df is not None and "time_hr" in df.columns) else set()

if TEST_DATE:
    d0 = pd.to_datetime(TEST_DATE).date()
    # All hourly stamps from MRoS that fall on TEST_DATE
    H = {h for h in hours(mros)  if pd.Timestamp(h).date()==d0}
    # All hourly stamps from IMERG that fall on TEST_DATE
    G = {h for h in hours(imerg) if (imerg is not None) and (pd.Timestamp(h).date()==d0)}
else:  # If no TEST_DATE, consider all available hours
    H = hours(mros)
    G = hours(imerg) if imerg is not None else set()

# Prefer an hour that exists in both MRoS and IMERG; if none, fall back to any MRoS hour.
candidates = (H & G) if (imerg is not None and len(G) > 0) else H
if not candidates:
    raise SystemExit("[STOP] No overlapping hour for requested day (or in data).")

# Choose hour with max MRoS points
counts = mros[mros["time_hr"].isin(candidates)].groupby("time_hr")["lon"].count()
sample_hour = counts.sort_values(ascending=False).index[0]
print(f"[INFO] sample_hour (UTC): {sample_hour}  (MRoS pts: {counts.loc[sample_hour]})")

In [ ]:
# ----------------------- Build aggregated set of points for PHASE IDW -------------------
# MRoS -> targets + DEM elev
mros_hr = mros.loc[mros["time_hr"]==sample_hour, ["lon","lat","phase"]].copy()
mros_hr["p_mix"]  = (mros_hr["phase"]=="mix").astype(float) # only MRoS provides info on "mix"
# Map MRoS phase to a snow probability: rain=0, snow=1, mix=0.5
mros_hr["p_snow"] = np.where(mros_hr["phase"]=="snow", 1.0,
                      np.where(mros_hr["phase"]=="rain", 0.0, 0.5)).astype(float)
mros_hr["alpha"]  = ALPHA_MROS
mros_hr["src"]    = "mros"
mros_hr["elev_meta"] = np.nan
# Sample DEM elevation at each MRoS point
mros_hr["elev_dem"] = sample_elev_points(PATHS["elev_tif"], mros_hr["lon"].to_numpy(), mros_hr["lat"].to_numpy())

pts_list = [mros_hr[["lon","lat","p_snow","p_mix","alpha","src","elev_dem", "elev_meta"]]]

# IMERG -> p_snow + DEM elev
imerg_hr = None
if imerg is not None:
    imerg_hr = imerg.loc[imerg["time_hr"]==sample_hour, ["lon","lat","p_liquid"]].copy()
    if not imerg_hr.empty:
        # Convert IMERG liquid probability to snow probability
        imerg_hr["p_snow"] = (1.0 - imerg_hr["p_liquid"]).clip(0,1).astype(float)
        # IMERG has no 'mix' concept → set to 0
        imerg_hr["p_mix"]  = 0.0
        imerg_hr["alpha"]  = ALPHA_IMERG
        imerg_hr["src"]    = "imerg"
        imerg_hr["elev_meta"] = np.nan
        # Sample DEM elevation at each IMERG point
        imerg_hr["elev_dem"] = sample_elev_points(PATHS["elev_tif"], imerg_hr["lon"].to_numpy(), imerg_hr["lat"].to_numpy())
        pts_list.append(imerg_hr[["lon","lat","p_snow","p_mix","alpha","src","elev_dem", "elev_meta"]])

# Stations -> add points anchored to nearest IMERG p_snow; carry predictors and station elevation
st_hr = None
if (st_all is not None) and ("time_hr" in st_all.columns) and (imerg_hr is not None) and (not imerg_hr.empty):
    # Pull station locations + predictors for this hour
    st_hr = st_all.loc[st_all["time_hr"]==sample_hour, ["lon","lat","elev","temp_air","temp_dew","temp_wet","rh"]].copy()
    st_hr = st_hr.dropna(subset=["lon","lat"])
    if not st_hr.empty:
        # For each station location, grab the nearest IMERG snow probability (so stations help densify p_snow)
        tree = cKDTree(imerg_hr[["lon","lat"]].to_numpy())
        d, idx = tree.query(st_hr[["lon","lat"]].to_numpy(), k=1)
        st_hr["p_snow"] = imerg_hr["p_snow"].to_numpy()[idx]
        st_hr["p_mix"]  = 0.0                                      # stations do not report 'mix'; we don't invent it
        st_hr["alpha"]  = ALPHA_STN
        st_hr["src"]    = "station"
        st_hr["elev_dem"] = sample_elev_points(PATHS["elev_tif"], st_hr["lon"].to_numpy(), st_hr["lat"].to_numpy())
        pts_list.append(st_hr[["lon","lat","p_snow","p_mix","alpha","src","elev","elev_dem"]].rename(columns={"elev":"elev_meta"}))

# Concatenate MRoS + IMERG + station-anchored points into one table for the phase IDW
pts = pd.concat(pts_list, ignore_index=True)

In [ ]:
# ----------------------- PHASE IDW  -----------

# Coordinates (longitude, latitude) of each station.
XY = pts[["lon","lat"]].to_numpy()
# Recall weight factors for each dataset source that can down- or up-weight its influence in IDW
ALPHA = pts["alpha"].to_numpy().astype(float)

P_mix_raw  = idw_predict_grid_weighted(XY, pts["p_mix" ].to_numpy().astype(float), xs, ys, k=K_NEIGH, power=POWER, alpha=ALPHA).clip(0,1)
P_snow_raw = idw_predict_grid_weighted(XY, pts["p_snow"].to_numpy().astype(float), xs, ys, k=K_NEIGH, power=POWER, alpha=ALPHA).clip(0,1)

# Renormalize: after separate interpolations, P_mix_raw + P_snow_raw might not sum to ≤ 1... 
# Compute the "mass" available for P_snow after P_mix is allocated:
mass   = (1.0 - P_mix_raw)
# Scale P_snow_raw
P_snow = (P_snow_raw * mass).clip(0,1)
# assign P_rain as the leftover probability
P_rain = (1.0 - P_snow - P_mix_raw).clip(0,1)
P_mix  = P_mix_raw

# Combine the three grids into one array of shape (3, nx, ny).
cls_idx = np.argmax(np.stack([P_rain, P_snow, P_mix], axis=0), axis=0)

# We will keep separate IDW runs for each phase probability to maintain the shape of each predictor field


In [ ]:
# ----------------------- STATION PREDICTOR IDW  -----------
def idw_station_predictor(pts_df, colname):
    if pts_df is None or colname not in pts_df.columns: 
        return np.full((len(ys), len(xs)), np.nan)
    mask = (pts_df["src"]=="station") & (pts_df[colname].notna())
    if not mask.any():
        return np.full((len(ys), len(xs)), np.nan)
    XYs = pts_df.loc[mask, ["lon","lat"]].to_numpy()
    zs  = pd.to_numeric(pts_df.loc[mask, colname], errors="coerce").to_numpy()
    keep = np.isfinite(zs)
    if keep.sum()==0: 
        return np.full((len(ys), len(xs)), np.nan)
    return idw_predict_grid_weighted(XYs, zs[keep], xs, ys, k=K_NEIGH, power=POWER, alpha=None)

T_air_grid = idw_station_predictor(st_hr.rename(columns=str), "temp_air") if st_hr is not None else np.full((len(ys), len(xs)), np.nan)
T_dew_grid = idw_station_predictor(st_hr.rename(columns=str), "temp_dew") if st_hr is not None else np.full((len(ys), len(xs)), np.nan)
T_wet_grid = idw_station_predictor(st_hr.rename(columns=str), "temp_wet") if st_hr is not None else np.full((len(ys), len(xs)), np.nan)
RH_grid    = idw_station_predictor(st_hr.rename(columns=str), "rh")       if st_hr is not None else np.full((len(ys), len(xs)), np.nan)

In [ ]:
# ----------------------- Visualize  ---------------------

# cls_idx 
fig, ax = plt.subplots(figsize=(7,6))
cmap = ListedColormap(["tab:blue","tab:gray","tab:orange"])  # 0=rain,1=snow,2=mix
im = ax.imshow(cls_idx, origin="lower",
               extent=(xs.min(), xs.max(), ys.min(), ys.max()),
               cmap=cmap, vmin=0, vmax=2)
for name, c in [("mros","k"), ("imerg","w"), ("station","lime")]:
    df = pts[pts["src"]==name]
    if not df.empty: ax.scatter(df["lon"], df["lat"], s=8, edgecolors="none", c=c, alpha=0.6, label=name)
ax.legend(loc="lower left", fontsize=8, frameon=True)
ax.set_title(f"IDW fused phase (MRoS + IMERG + Stations), DEM bounds — {pd.Timestamp(sample_hour)} UTC")
cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, ticks=[0,1,2]); cbar.ax.set_yticklabels(["rain","snow","mix"])
plt.tight_layout(); plt.show()

# temp_wet predictor surface
fig2, ax2 = plt.subplots(figsize=(7,6))
im2 = ax2.imshow(T_wet_grid, origin="lower", extent=(xs.min(), xs.max(), ys.min(), ys.max()))
ax2.set_title(f"Station-only IDW — temp_wet @ {pd.Timestamp(sample_hour)} UTC")
plt.colorbar(im2, ax2=ax2, fraction=0.046, pad=0.04); plt.tight_layout(); plt.show()